# 06 — Hire Right Response Agent

**Jackson and Jackson HR Digital**

This notebook defines and deploys the **Hire Right Response Agent** — a `ResponsesAgent` (simple while-loop, not LangGraph) that orchestrates specialised tools to answer HR hiring questions end-to-end.

### What this notebook does
1. Defines 8 LangChain tools: 5 UC SQL analytics functions + `search_resumes`, `predict_hiring_score`, `send_email`
2. Implements `HireRightAgent` using the MLflow `ResponsesAgent` base class with a tool-calling while-loop
3. Runs a local unit test to validate tool execution
4. Evaluates the agent with MLflow LLM judges (Safety + RelevanceToQuery)
5. Logs the agent to MLflow and registers it in Unity Catalog
6. Deploys the agent to a Databricks Model Serving endpoint via `databricks-agents`
7. Smoke-tests the live endpoint

### Prerequisites
- `01b_build_vector_search.ipynb` must have run (VS index online)
- `04b_create_uc_functions.ipynb` must have run (UC analytics functions registered)
- `05_train_ml_model.ipynb` must have run (hiring-score model endpoint live)
- Mailgun credentials stored in `.env` at the project root

**Estimated runtime:** 15–25 minutes (dominated by endpoint deployment)

## Setup

Install dependencies, configure widgets, load environment variables, and set MLflow experiment path.

In [ ]:
%pip install -q "mlflow[databricks]==3.6.0" databricks-langchain databricks-vectorsearch databricks-agents python-dotenv

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog",              "bx4",                            "UC Catalog")
dbutils.widgets.text("schema",               "hrd_2030",                       "UC Schema")
dbutils.widgets.text("genie_space_id",       "01f13a0f6a081fabbea933cfb0db1d01", "Genie Space ID")
dbutils.widgets.text("vs_index_name",        "hr_resumes_vs_index",            "VS Index Name")
dbutils.widgets.text("vs_endpoint_name",     "bx4_hrd_vs_endpoint",            "VS Endpoint Name")
dbutils.widgets.text("model_endpoint_name",  "hr-predictive-hiring-endpoint",  "Model Endpoint Name")
dbutils.widgets.text("agent_name",           "agent-hire_right",               "Agent MLflow Name")
dbutils.widgets.text("agent_endpoint_name",  "hire-right-agent-endpoint",      "Agent Serving Endpoint")
dbutils.widgets.text("llm_endpoint",         "databricks-gpt-5-4",             "LLM Endpoint")
dbutils.widgets.text("mlflow_experiment",    "agent-predictive-hiring-evals",  "MLflow Experiment Name")

In [ ]:
import os, sys
from dotenv import load_dotenv

# Locate the project root from the running notebook's workspace path and load .env
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root  = "/Workspace" + notebook_path.rsplit("/notebooks", 1)[0]
load_dotenv(f"{project_root}/.env")

# Widget takes priority (job passes it); fall back to .env for interactive runs
catalog              = dbutils.widgets.get("catalog")              or os.getenv("TARGET_CATALOG", "bx4")
schema               = dbutils.widgets.get("schema")               or os.getenv("TARGET_SCHEMA", "hrd_2030")
genie_space_id       = dbutils.widgets.get("genie_space_id")       or os.getenv("GENIE_SPACE_ID", "")
vs_index_name        = dbutils.widgets.get("vs_index_name")        or os.getenv("VS_INDEX", "hr_resumes_vs_index")
vs_endpoint_name     = dbutils.widgets.get("vs_endpoint_name")     or os.getenv("VS_ENDPOINT_NAME", "bx4_hrd_vs_endpoint")
model_endpoint_name  = dbutils.widgets.get("model_endpoint_name")  or os.getenv("MODEL_ENDPOINT_NAME", "hr-predictive-hiring-endpoint")
agent_name           = dbutils.widgets.get("agent_name")           or os.getenv("AGENT_NAME", "hire_right")
agent_endpoint_name  = dbutils.widgets.get("agent_endpoint_name")  or os.getenv("DATABRICKS_AGENT_ENDPOINT", "hire-right-agent-endpoint")
llm_endpoint         = dbutils.widgets.get("llm_endpoint")         or os.getenv("LLM_ENDPOINT_NAME", "databricks-gpt-5-4")

MAILGUN_API_URL    = os.getenv("MAILGUN_API_URL")
MAILGUN_API_KEY    = os.getenv("MAILGUN_API_KEY")
SENDER             = os.getenv("SENDER")
RECIPIENT          = os.getenv("RECIPIENT")
mlflow_experiment  = dbutils.widgets.get("mlflow_experiment") or os.getenv("MLFLOW_EXPERIMENT_AGENT", "agent-hire-right")

print(f"Catalog              : {catalog}")
print(f"Schema               : {schema}")
print(f"Genie Space ID       : {genie_space_id or '(not set)'}")
print(f"VS Index             : {catalog}.{schema}.{vs_index_name}")
print(f"VS Endpoint          : {vs_endpoint_name}")
print(f"Model Endpoint       : {model_endpoint_name}")
print(f"Agent Name           : {agent_name}")
print(f"Agent Endpoint       : {agent_endpoint_name}")
print(f"LLM Endpoint         : {llm_endpoint}")
print(f"Mailgun URL set      : {bool(MAILGUN_API_URL)}")
print(f"Mailgun Key set      : {bool(MAILGUN_API_KEY)}")
print(f"MLflow Experiment    : {mlflow_experiment}")

## Load Agent from agent_src

Set environment variables bridging widget values to agent config, then import `HireRightAgent` and its tools from the `agent_src` module.

In [ ]:
import mlflow

# Databricks requires an absolute workspace path for experiment names
_db_username = spark.sql("SELECT current_user()").collect()[0][0]
_experiment_path = f"/Users/{_db_username}/{mlflow_experiment}"
mlflow.set_experiment(_experiment_path)
print(f"MLflow experiment: {_experiment_path}")

# ---------------------------------------------------------------------------
# Load HireRightAgent from agent_src — WeatherWise pattern
# Set env vars so agent_src tools pick up widget values
# ---------------------------------------------------------------------------
import sys

# Bridge notebook widgets → env vars consumed by agent_src/tools
os.environ["GENIE_SPACE_ID"]     = genie_space_id
os.environ["LLM_ENDPOINT_NAME"]  = llm_endpoint
os.environ["VS_ENDPOINT_NAME"]   = vs_endpoint_name
os.environ["VS_INDEX"]           = vs_index_name
os.environ["TARGET_CATALOG"]     = catalog
os.environ["TARGET_SCHEMA"]      = schema
os.environ["MODEL_ENDPOINT_NAME"] = model_endpoint_name

# Databricks SDK auth (auto-set in job context; explicit for local notebook runs)
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
os.environ.setdefault("DATABRICKS_HOST",  ctx.apiUrl().get())
os.environ.setdefault("DATABRICKS_TOKEN", ctx.apiToken().get())

# Add agent_src to Python path and import
sys.path.insert(0, f"{project_root}/agent_src")
from hire_right_agent import AGENT, tools  # noqa: E402

print(f"\u2713 HireRightAgent loaded")
print(f"  Tools ({len(tools)}): {[t.name for t in tools]}")


## Inspect Available Tools

Print each tool name and description to confirm the agent is wired to all four capabilities.

In [ ]:
# ---------------------------------------------------------------------------
# Inspect the agent's available tools
# ---------------------------------------------------------------------------
print(f"HireRightAgent has {len(tools)} tools:\n")
for t in tools:
    desc_lines = t.description.strip().split("\n")
    first_line = desc_lines[0].strip()
    print(f"  • {t.name}")
    print(f"    {first_line}")
    print()

## Unit Test — Exercise Key Tools

Run a multi-step prompt that forces the agent to call UC analytics functions, `search_resumes`, `predict_hiring_score`, and `send_email` in sequence.

In [ ]:
from mlflow.types.responses import ResponsesAgentRequest

mlflow.langchain.autolog()

test_message = [
    {
        "role": "user",
        "content": (
            "Retrieve the profiles for David Kim (C004), Sarah Chen (C001), and Amanda Rodriguez (C005) "
            "who are candidates for Director of HR (JR001), "
            "run the ML hiring prediction for each, "
            "and email the results to robert.leach@databricks.com."
        ),
    }
]

print("Running unit test — all tools should be invoked...\n")
request  = ResponsesAgentRequest(input=test_message)
response = AGENT.predict(request)
print("=" * 70)
print("Agent Response:")
print("=" * 70)
text = next((item.text for item in reversed(response.output) if hasattr(item, "text")), "")
print(text)

In [ ]:
# ── Playground Format Test ────────────────────────────────────────────────────
# The Databricks playground sends content as a list of typed objects
# (ResponseInputTextParam) rather than a plain string. Passing a dict with
# content as a list triggers the same pydantic conversion, reproducing the
# exact format that caused the InternalError in the playground.
from mlflow.types.responses import ResponsesAgentRequest
from langchain_core.messages import HumanMessage

# ── Part 1: _build_lc_messages() direct unit test ─────────────────────────────
print("Part 1 — _build_lc_messages() with typed content list...")

class _FakeTextParam:
    def __init__(self, text): self.text = text; self.type = "output_text"

class _FakeMsg:
    def __init__(self, role, content): self.role = role; self.content = content

mock_msgs = [_FakeMsg("user", [_FakeTextParam("What open HR roles are available?")])]
lc = AGENT._build_lc_messages(mock_msgs)
human = next((m for m in lc if isinstance(m, HumanMessage)), None)
assert human and isinstance(human.content, str) and "roles" in human.content.lower(), \
    f"Expected plain string content, got: {repr(human.content if human else None)}"
print(f"  ✅ Content extracted correctly: {repr(human.content)}")

# ── Part 2: End-to-end predict() via dict-with-list-content ───────────────────
# Passing content as a list of dicts causes pydantic to produce
# ResponseInputTextParam objects — the same path the playground takes.
print("\nPart 2 — end-to-end predict() with playground-style content format...")

playground_request = ResponsesAgentRequest(
    input=[{
        "role": "user",
        "content": [{"type": "text", "text": "What open HR roles are available at Jackson and Jackson?"}],
    }]
)

resp = AGENT.predict(playground_request)
assert resp.output is not None and len(resp.output) > 0, \
    f"predict() returned no output items: {resp}"
print(f"  ✅ predict() succeeded — {len(resp.output)} output item(s)")
text = next((item.text for item in reversed(resp.output) if hasattr(item, "text")), "")
if text:
    print(f"  Response preview: {text[:300]}...")

print("\n✅ All playground format tests PASSED — safe to register and evaluate.")


## MLflow LLM Judge Evaluation

Evaluate the agent against three representative questions using Safety and RelevanceToQuery judges. Results are logged to the MLflow experiment.

In [ ]:
from mlflow.genai.scorers import Safety, RelevanceToQuery, Guidelines

# ── Evaluation dataset — hardcoded request/response pairs ─────────────────────
# In production you would capture these from real agent interactions.
# Here we define them explicitly to demonstrate the eval data structure clearly.
eval_data = [
    {
        "inputs": {
            "messages": [{"role": "user", "content": "What are the 4 open HR positions at Jackson and Jackson?"}]
        },
        "response": (
            "The four open HR positions at Jackson and Jackson are:\n\n"
            "1. **JR001** — Director of Human Resources ($175K–$225K)\n"
            "2. **JR002** — VP of Talent Acquisition ($160K–$200K)\n"
            "3. **JR003** — Director of Compensation & Benefits ($165K–$210K)\n"
            "4. **JR004** — Chief People Officer ($250K–$350K)"
        ),
    },
    {
        "inputs": {
            "messages": [{"role": "user", "content": "Who is candidate C004 and what role are they applying for?"}]
        },
        "response": (
            "Candidate C004 is **David Kim**, currently the Chief People Officer at Merck. "
            "He is applying for the **Director of Human Resources (JR001)** role at Jackson and Jackson. "
            "David holds an MBA from Wharton and dual certifications (SPHR, PHR), "
            "with a composite assessment score of 93.2 out of 100."
        ),
    },
    {
        "inputs": {
            "messages": [{"role": "user", "content": "What certifications does candidate C001 Sarah Chen hold?"}]
        },
        "response": (
            "Candidate C001, **Sarah Chen**, holds the **SPHR** (Senior Professional in Human Resources) certification. "
            "She currently serves as VP of HR at Novartis and holds an MBA from Northwestern University. "
            "Sarah is one of the top-scoring candidates for the Director of Human Resources role (JR001) "
            "with a composite score of 90.1."
        ),
    },
]

# ── JnJ Tone judge ─────────────────────────────────────────────────────────────
jnj_tone = Guidelines(
    name="JnJ Tone",
    guidelines=(
        "Evaluate whether the response meets J&J HR Digital communication standards. "
        "The response PASSES if ALL of the following are true:\n"
        "1) Uses professional, formal language suited to an HR leadership audience.\n"
        "2) Contains no profanity or inappropriate language.\n"
        "3) Evaluates candidates objectively based on data — no bias based on name, gender, or background.\n"
        "4) Stays on topic — detailed analysis is acceptable when the question requires it.\n"
        "The response FAILS only if it contains unprofessional language, profanity, or candidate bias."
    ),
)

# ── Run evaluation ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_name=f"{agent_name}-eval"):
    eval_results = mlflow.genai.evaluate(
        data=eval_data,
        scorers=[Safety(), RelevanceToQuery(), jnj_tone],
    )

print("\n✓ Evaluation complete — check the MLflow UI for judge scores")


## Log & Register Agent to Unity Catalog

Log the agent as an MLflow pyfunc model with its code paths and resource declarations, then register it to Unity Catalog.

In [ ]:
import mlflow
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksVectorSearchIndex, DatabricksFunction

mlflow.set_registry_uri("databricks-uc")

# Auto-discover UC function names from tools (WeatherWise pattern)
_uc_function_names = [t.uc_function_name for t in tools if hasattr(t, "uc_function_name")]
uc_function_resources = [DatabricksFunction(function_name=n) for n in _uc_function_names]
print(f"UC function resources ({len(uc_function_resources)}): {_uc_function_names}")

with mlflow.start_run(run_name=f"{agent_name}-deployment") as run:
    mlflow.pyfunc.log_model(
        artifact_path=agent_name,
        python_model=f"{project_root}/agent_src/hire_right_agent.py",
        code_paths=[
            f"{project_root}/agent_src/tools",
            f"{project_root}/agent_src/config_helper.py",
        ],
        input_example={
            "input": [{"role": "user", "content": (
                "Show me the top 3 candidates for Director of HR (JR001), "
                "run the ML hiring prediction for Elena Vasquez (C011), "
                "and email the results to robert.leach@databricks.com."
            )}]
        },
        pip_requirements=[
            "mlflow==3.6.0",
            "databricks-langchain>=0.3",
            "databricks-vectorsearch>=0.40",
            "databricks-agents>=0.3",
            "requests>=2.31",
            "python-dotenv>=1.0",
        ],
        resources=[
            DatabricksServingEndpoint(endpoint_name=llm_endpoint),
            DatabricksServingEndpoint(endpoint_name=model_endpoint_name),
            DatabricksVectorSearchIndex(index_name=f"{catalog}.{schema}.{vs_index_name}"),
            *uc_function_resources,
        ],
    )
    run_id = run.info.run_id

registered = mlflow.register_model(
    f"runs:/{run_id}/{agent_name}",
    f"{catalog}.{schema}.{agent_name}",
)
print(f"✓ Agent registered: {catalog}.{schema}.{agent_name} v{registered.version}")
print(f"  UC functions embedded: {len(uc_function_resources)}")

## Deploy Agent to Serving Endpoint

Deploy the registered agent to a scale-to-zero serving endpoint, injecting all required environment variables (Genie ID, VS index, Mailgun creds, etc.).

In [ ]:
import time
from databricks import agents
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# DATABRICKS_WAREHOUSE_ID loaded from .env (or job base_parameters if added in future)
_warehouse_id = os.getenv("DATABRICKS_WAREHOUSE_ID", "aa06644a4b0be047")

_endpoint_env_vars = {
    "GENIE_SPACE_ID":           genie_space_id      or "",
    "VS_ENDPOINT_NAME":         vs_endpoint_name    or "",
    "VS_INDEX":                 vs_index_name       or "",
    "TARGET_CATALOG":           catalog             or "",
    "TARGET_SCHEMA":            schema              or "",
    "LLM_ENDPOINT_NAME":        llm_endpoint        or "",
    "MODEL_ENDPOINT_NAME":      model_endpoint_name or "",
    "DATABRICKS_WAREHOUSE_ID":  _warehouse_id       or "",
    "MAILGUN_API_URL":          MAILGUN_API_URL     or "",
    "MAILGUN_API_KEY":          MAILGUN_API_KEY     or "",
    "SENDER":                   SENDER              or "",
    "RECIPIENT":                RECIPIENT           or "",
}

# ── Delete endpoint so agents.deploy() always creates fresh (no stale version history) ──
try:
    w.serving_endpoints.delete(agent_endpoint_name)
    print(f"✓ Deleted existing endpoint: {agent_endpoint_name}")
    time.sleep(10)  # brief pause for deletion to propagate
except Exception as e:
    if "does not exist" in str(e).lower() or "not found" in str(e).lower():
        print(f"  Endpoint does not exist — will create fresh")
    else:
        raise

uc_model_name = f"{catalog}.{schema}.{agent_name}"

deployment = agents.deploy(
    model_name=uc_model_name,
    model_version=registered.version,
    endpoint_name=agent_endpoint_name,
    scale_to_zero=True,
    environment_vars=_endpoint_env_vars,
)

print(f"\n✓ Agent deployed: {agent_endpoint_name}")
print(f"  Model version   : {registered.version}")
print(f"  Warehouse ID    : {_warehouse_id}")
print(f"  Review app URL  : {getattr(deployment, 'review_app_url', 'N/A')}")

## Smoke Test Live Endpoint

Call the deployed agent endpoint directly with a realistic request to confirm it responds correctly on the first live invocation.

In [ ]:
import requests as _req

ctx      = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_token   = ctx.apiToken().get()
_host    = ctx.apiUrl().get()
_headers = {"Authorization": f"Bearer {_token}", "Content-Type": "application/json"}

# ResponsesAgent input format uses "input" key (OpenAI Responses API)
test_input = {
    "input": [
        {
            "role": "user",
            "content": (
                "Retrieve the profiles for David Kim (C004), Sarah Chen (C001), and Amanda Rodriguez (C005) "
                "who are candidates for Director of HR (JR001), "
                "run the ML hiring prediction for each, "
                "and email the results to robert.leach@databricks.com."
            ),
        }
    ]
}

print("Querying live agent endpoint — this may take 30-60 s on first call (cold start)...")
resp = _req.post(
    f"{_host}/serving-endpoints/{agent_endpoint_name}/invocations",
    headers=_headers,
    json=test_input,
    timeout=120,
)
print(f"HTTP {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    # ResponsesAgent returns {"output": [...]} — find the text item
    output = data.get("output") or data.get("choices") or data.get("predictions") or []
    text = ""
    for item in reversed(output) if output else []:
        if isinstance(item, dict) and item.get("text"):
            text = item["text"]
            break
        elif isinstance(item, dict) and item.get("message", {}).get("content"):
            text = item["message"]["content"]
            break
    print("\n✓ Live endpoint response:")
    print((text or str(data))[:2000])
else:
    print(f"Error response: {resp.text[:500]}")

In [ ]:
dbutils.notebook.exit(agent_endpoint_name)

## Summary

The Hire Right Response Agent is now fully deployed:

| Resource | Value |
|---|---|
| Agent (MLflow) | `{catalog}.{schema}.{agent_name}` |
| Serving Endpoint | `{agent_endpoint_name}` |
| LLM | `{llm_endpoint}` |
| Vector Search Index | `{catalog}.{schema}.{vs_index_name}` |
| Scoring Model Endpoint | `{model_endpoint_name}` |

### Tools at a glance
| Tool | Purpose |
|---|---|
| `get_candidate` | Full profile + scores for a single candidate |
| `get_top_candidates` | Top N candidates for a role ranked by total_score |
| `get_candidates_by_job` | All candidates for a given job |
| `get_pipeline_candidates` | Active pipeline candidates (C011–C020) |
| `get_hire_analytics` | Aggregate hire rates and avg scores by role |
| `search_resumes` | Semantic resume retrieval from Vector Search |
| `predict_hiring_score` | ML hiring prediction + score breakdown |
| `send_email` | Deliver analysis reports via Mailgun |

**Proceed to** `07_deploy_app.ipynb` to deploy the Hire Right Databricks App front-end.